# Alternative feature-selection setup - 01 fold plan and recipe space

This notebook starts an alternative path to the classic `feature_selection.ipynb` workflow.

The classic path is useful for fast exploration: it ranks and narrows features early, then modeling notebooks reuse those feature sets. This workflow is stricter and more modular. It first declares the fold plan and the candidate recipe space, then later notebooks fit prescreening and models only inside the training partition allowed for each evaluation step.

The goal is to reduce optimistic validation from selecting features on too much target information. This notebook does not train models and does not select features. It only writes reusable artifacts that the later alternative-selection notebooks can read without rebuilding the split plan.

The central rule is simple:

- final assessment folds are used only for final fold scoring,
- tuning folds are used only for recipe scoring,
- every prescreening fit happens only on the active training partition.


## 0. Setup

The output directory for this first stage is `outputs/feature_selection_alternative/01_fold_plan_and_recipe_space/`. Later alternative notebooks read this exact path, so this setup stage does not use legacy output fallbacks.


In [ ]:
import importlib.util

import pandas as pd

from cost_effective.dataset import find_project_root, load_test_data, load_training_data
from cost_effective.models.feature_selection_alternative_setup import (
    DEFAULT_ALTERNATIVE_FEATURE_SIZES,
    DEFAULT_ALTERNATIVE_PRESCREEN_METHODS,
    assert_split_integrity,
    build_feature_size_grid,
    build_model_spec_space,
    build_pipeline_recipe_space,
    build_prescreen_recipe_space,
    leakage_contract_table,
    make_inner_fold_assignments,
    make_outer_fold_assignments,
    summarize_inner_folds,
    summarize_outer_folds,
    write_stage_one_outputs,
)

project_root = find_project_root()
outputs = (
    project_root / "outputs" / "feature_selection_alternative" / "01_fold_plan_and_recipe_space"
)
outputs.mkdir(parents=True, exist_ok=True)

X_train, y_train = load_training_data(project_root / "data")
X_test = load_test_data(project_root / "data")

X_train.shape, y_train.shape, X_test.shape

In [ ]:
RANDOM_STATE = 42
OUTER_SPLITS = 5
INNER_SPLITS = 5

PRESCREEN_METHODS = DEFAULT_ALTERNATIVE_PRESCREEN_METHODS
FEATURE_SIZES = DEFAULT_ALTERNATIVE_FEATURE_SIZES

RUN_XGBOOST = importlib.util.find_spec("xgboost") is not None
RUN_EBM = importlib.util.find_spec("interpret") is not None
RUN_LAMBDAMART = True

config = {
    "random_state": RANDOM_STATE,
    "outer_splits": OUTER_SPLITS,
    "inner_splits": INNER_SPLITS,
    "prescreen_methods": PRESCREEN_METHODS,
    "feature_sizes": FEATURE_SIZES,
    "run_xgboost": RUN_XGBOOST,
    "run_ebm": RUN_EBM,
    "run_lambdamart": RUN_LAMBDAMART,
    "notes": "Stage 1 defines the fold plan and recipe space for the alternative feature-selection workflow; no feature selection is fitted here.",
}

pd.Series(config)

## 1. Create final assessment folds

These folds produce the final validation estimate for the alternative path. Later notebooks must not use an assessment fold for feature selection, model selection, or HPO.


In [ ]:
outer_assignments = make_outer_fold_assignments(
    y_train,
    n_splits=OUTER_SPLITS,
    random_state=RANDOM_STATE,
)
outer_summary = summarize_outer_folds(y_train, outer_assignments)

outer_assignments.head(), outer_summary

## 2. Create tuning folds inside each training partition

For each assessment fold, tuning CV is created only from the corresponding training samples. The assessment samples for that fold do not appear in its tuning table.


In [ ]:
inner_assignments = make_inner_fold_assignments(
    y_train,
    outer_assignments,
    n_splits=INNER_SPLITS,
    random_state=RANDOM_STATE,
)
inner_summary = summarize_inner_folds(y_train, inner_assignments)

assert_split_integrity(
    outer_assignments,
    inner_assignments,
    n_samples=len(y_train),
    outer_splits=OUTER_SPLITS,
    inner_splits=INNER_SPLITS,
)

inner_assignments.head(), inner_summary.head(10)

## 3. Define the data-access contract

This table is the contract for all later notebooks. If a later stage violates it, the validation estimate is no longer meaningful.


In [ ]:
leakage_contract = leakage_contract_table()
leakage_contract

## 4. Define prescreening recipes

These are declarations, not fitted rankings. Adaptive weighted recipes are allowed only if their weights are estimated inside the active training partition in later stages.


In [ ]:
prescreen_recipes = build_prescreen_recipe_space(PRESCREEN_METHODS)
prescreen_recipes.groupby(["prescreen_method_count", "adaptive"]).size().rename(
    "n_recipes"
).reset_index()

In [ ]:
prescreen_recipes.head(20)

## 5. Define feature-size and model/HPO spaces

Feature size means top-k features after a fold-local prescreening ranking. Model specs are also only declarations here. Later stages will instantiate and fit them only inside the correct data boundaries.


In [ ]:
feature_sizes = build_feature_size_grid(FEATURE_SIZES)
model_specs = build_model_spec_space(
    include_xgboost=RUN_XGBOOST,
    include_ebm=RUN_EBM,
    include_lambdamart=RUN_LAMBDAMART,
)

model_specs.groupby(["base_model_family", "model_kind"]).size().rename("n_specs").reset_index()

In [ ]:
feature_sizes, model_specs.head(20)

## 6. Build the declarative pipeline recipe space

This is the full Cartesian declaration of:

```text
prescreen recipe x feature size x model spec
```

It is intentionally not a command to train every row blindly. Later stages can use fold-local pruning inside tuning CV, but any pruning must happen only using data allowed by the data-access contract.


In [ ]:
pipeline_recipes = build_pipeline_recipe_space(
    prescreen_recipes,
    feature_sizes,
    model_specs,
)

recipe_space_overview = pd.Series({
    "n_prescreen_recipes": len(prescreen_recipes),
    "n_feature_sizes": len(feature_sizes),
    "n_model_specs": len(model_specs),
    "n_pipeline_recipes": len(pipeline_recipes),
})
recipe_space_overview

In [ ]:
pipeline_recipes.head(20)

## 7. Persist stage-one artifacts

The next notebooks should read these files instead of rebuilding folds ad hoc. That keeps the alternative feature-selection workflow stable and auditable.


In [ ]:
write_stage_one_outputs(
    output_dir=outputs,
    outer_assignments=outer_assignments,
    outer_summary=outer_summary,
    inner_assignments=inner_assignments,
    inner_summary=inner_summary,
    prescreen_recipes=prescreen_recipes,
    feature_sizes=feature_sizes,
    model_specs=model_specs,
    pipeline_recipes=pipeline_recipes,
    config=config,
)

output_manifest = pd.DataFrame([
    {"file": "stage_config.json", "meaning": "Stage-one configuration."},
    {"file": "outer_fold_assignments.csv", "meaning": "One outer validation fold per sample."},
    {"file": "outer_fold_summary.csv", "meaning": "Outer fold class-balance summary."},
    {
        "file": "inner_fold_assignments.csv",
        "meaning": "Inner validation folds inside each outer train split.",
    },
    {"file": "inner_fold_summary.csv", "meaning": "Inner fold class-balance summary."},
    {"file": "leakage_contract.csv", "meaning": "Allowed and forbidden data access per stage."},
    {
        "file": "prescreen_recipe_space.csv",
        "meaning": "Fold-local prescreening recipe declarations.",
    },
    {"file": "feature_size_grid.csv", "meaning": "Top-k feature-size declarations."},
    {"file": "model_spec_space.csv", "meaning": "Model/HPO specification declarations."},
    {"file": "pipeline_recipe_space.csv", "meaning": "Declarative full pipeline recipe space."},
    {
        "file": "pipeline_recipe_space_summary.csv",
        "meaning": "Recipe-count summary by model family and feature size.",
    },
    {"file": "experiment_log.csv", "meaning": "Empty stable log table for later stages."},
])
output_manifest.to_csv(outputs / "output_manifest.csv", index=False)
output_manifest

## 8. Sanity checks for later stages

These checks are intentionally simple and should stay true throughout the alternative workflow.


In [ ]:
sanity_checks = pd.Series({
    "outer_assignment_rows": len(outer_assignments),
    "unique_outer_samples": outer_assignments["sample_index"].nunique(),
    "outer_fold_count": outer_assignments["outer_fold"].nunique(),
    "inner_outer_fold_count": inner_assignments["outer_fold"].nunique(),
    "inner_fold_count_per_outer_min": inner_assignments.groupby("outer_fold")["inner_fold"]
    .nunique()
    .min(),
    "inner_fold_count_per_outer_max": inner_assignments.groupby("outer_fold")["inner_fold"]
    .nunique()
    .max(),
    "pipeline_recipe_rows": len(pipeline_recipes),
})
san = sanity_checks.to_frame("value")
san.to_csv(outputs / "stage_one_sanity_checks.csv")
san